In [2]:
import os 
import numpy as np
import pandas as pd 

In [25]:
directory = "final_cleaned"
files = list(filter(lambda x : "~" not in x or "hdfc" in x.lower(), os.listdir(directory)))
types = np.array([]).astype(str)

In [26]:
for file in files:
    types = np.append(types , pd.read_excel(os.path.join(directory,file))["Type"].unique())

In [27]:
len(np.unique(types))

232

In [3]:
from rapidfuzz import fuzz

# Example:

fuzz.ratio("hello world", "hello world scheme")  # high score


75.86206896551724

In [15]:
from sentence_transformers import SentenceTransformer
import hdbscan

texts = ["- hello world ---"
, "hello world scheme"
, "hello world plan"
, "equity"
, "equity related securities"
, "equity and debt mix"
, "foreign investments"]
model = SentenceTransformer('nli-roberta-large')
embeddings = model.encode(texts)

clusterer = hdbscan.HDBSCAN(min_cluster_size=2)
labels = clusterer.fit_predict(embeddings)


In [32]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import wordninja

# Download once
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    tokens = re.findall(r'[a-zA-Z]+', text.lower())
    split_tokens = []
    for token in tokens:
        if len(token) > 15:  # heuristic: likely a compound
            split_tokens.extend(wordninja.split(token))
        else:
            split_tokens.append(token)
    return " ".join(split_tokens)

from sentence_transformers import SentenceTransformer

entries = [clean_text(t) for t in types]

model = SentenceTransformer('nli-roberta-large')  # Small and fast
embeddings = model.encode(entries)

import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
labels = clusterer.fit_predict(embeddings)

from collections import defaultdict, Counter

cluster_map = defaultdict(list)

for label, entry in zip(labels, entries):
    if label != -1:
        cluster_map[label].append(entry)

representatives = {
    label: min(texts, key=len)  # or use frequency / centrality
    for label, texts in cluster_map.items()
}

# Assuming `types` is your original input list
final_mapping = []

for original, cleaned, label in zip(types, entries, labels):
    if label == -1:
        category = "uncategorized"
    else:
        category = representatives[label]
    final_mapping.append((original, cleaned, category))

# Print results in table format
print(f"{'Original Entry':40} | {'Cleaned Entry':35} | {'Final Label'}")
print("-" * 100)
for orig, clean, label in final_mapping:
    print(f"{orig:40} | {clean:35} | {label}")



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vaibh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vaibh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vaibh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Original Entry                           | Cleaned Entry                       | Final Label
----------------------------------------------------------------------------------------------------
REIT/InvIT Instruments                   | reit invit instruments              | uncategorized
Debt Instruments GOI                     | debt instruments goi                | uncategorized
Corporate Debt Market Development Fund   | corporate debt market development fund | state development loan
Debt Instruments                         | debt instruments                    | debt instruments
Certificate of Deposit                   | certificate of deposit              | certificate of deposit
Commercial Paper PUBA                    | commercial paper puba               | uncategorized
Treasury Bill                            | treasury bill                       | treasury bill
Equity & Equity related                  | equity equity related               | equity equity related
 Unlisted     

In [43]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', '', text) 
    tokens = re.findall(r'[a-zA-Z]+', text.lower())
    split_tokens = []
    for token in tokens:
        split_tokens.extend(wordninja.split(token))
    return " ".join(split_tokens)

# Step 1: Clean entries
cleaned_map = {e: clean_text(e) for e in np.unique(types)}

# Step 2: Match entries
categories = {}
representatives = []

SPECIAL_EXCLUSIONS = {'uncategorized', 'unlisted'}

for entry, cleaned in cleaned_map.items():
    if cleaned in SPECIAL_EXCLUSIONS:
        categories[entry] = cleaned  # map only to itself
        continue

    matched = False
    for rep in representatives:
        score = fuzz.ratio(cleaned, clean_text(rep))
        if score >= 95:
            categories[entry] = rep
            matched = True
            break

    if not matched:
        # New category
        categories[entry] = entry
        representatives.append(entry)

# Step 3: Print final mappings
print(f"{'Original Entry':40} | {'Category'}")
print("-" * 60)
for entry in types:
    print(f"{entry:40} | {categories.get(entry, '⚠️ not found')}")


Original Entry                           | Category
------------------------------------------------------------
REIT/InvIT Instruments                   | REIT/InvIT Instruments 
Debt Instruments GOI                     | Debt Instruments GOI 
Corporate Debt Market Development Fund   | Corporate Debt Market Development Fund 
Debt Instruments                         |  DEBT INSTRUMENTS 
Certificate of Deposit                   |  Certificate of Deposit 
Commercial Paper PUBA                    | Commercial Paper PUBA 
Treasury Bill                            |  Treasury Bill 
Equity & Equity related                  |  EQUITY & EQUITY RELATED 
 Unlisted                                | unlisted
 CMSI                                    |  CMSI 
uncategorised                            | uncategorised
Gold                                     |  Gold 
Equity & Equity related                  |  EQUITY & EQUITY RELATED 
Foreign Securities and/or overseas ETF   |  Foreign Securities and/or 

In [44]:
import wordninja
import re
from rapidfuzz import fuzz

SPECIAL_EXCLUSIONS = {"uncategorized", "unlisted"}

def clean_text(text):
    text = re.sub(r'[^a-zA-Z]', '', text)
    text = text.lower()
    split_tokens = [t for t in wordninja.split(text) if len(t) > 1]
    return " ".join(split_tokens)

def choose_final_label(cleaned_text):
    # Strip leading "b" if it's noise (like in BUNITS)
    tokens = cleaned_text.split()
    if tokens and tokens[0] == 'b':
        return " ".join(tokens[1:])
    return cleaned_text

def map_entries_to_labels(entries, threshold=95):
    cleaned_entries = {e: clean_text(e) for e in entries}
    categories = {}
    canonical_labels = {}

    for entry, cleaned in cleaned_entries.items():
        if cleaned in SPECIAL_EXCLUSIONS:
            categories[entry] = entry
            canonical_labels[entry] = entry
            continue

        matched = False
        for rep, rep_cleaned in canonical_labels.items():
            score = fuzz.ratio(cleaned, clean_text(rep_cleaned))
            if score >= threshold:
                categories[entry] = rep
                matched = True
                break

        if not matched:
            # New category
            label = choose_final_label(cleaned)
            categories[entry] = label
            canonical_labels[entry] = label

    return [(entry, cleaned_entries[entry], categories[entry]) for entry in entries]


In [45]:
entries = types.copy()
results = map_entries_to_labels(entries)

print(f"{'Original Entry':45} | {'Cleaned Entry':45} | Final Label")
print("-" * 120)
for orig, clean, label in results:
    print(f"{orig:45} | {clean:45} | {label}")

Original Entry                                | Cleaned Entry                                 | Final Label
------------------------------------------------------------------------------------------------------------------------
REIT/InvIT Instruments                        | re it in vit instruments                      | re it in vit instruments
Debt Instruments GOI                          | debt instruments goi                          | debt instruments goi
Corporate Debt Market Development Fund        | corporate debt market development fund        | corporate debt market development fund
Debt Instruments                              | debt instruments                              | debt instruments
Certificate of Deposit                        | certificate of deposit                        | certificate of deposit
Commercial Paper PUBA                         | commercial paper pub                          | commercial paper pub
Treasury Bill                                 | t

In [46]:
ETF                                           | et                                            | et
CMSI                                         | cms                                           | cms
ReIT                                          | re it                                         | re it
InvIT                                         | in vit                                        | in vit
BUNLISTED                                     | bun listed                                    | bun listed
Securitized Debt Instruments                 | se curit zed debt instruments                 | SECURITIZEDDEBTINSTRUMENTS 
CNX NIFTY-JUN - - -                          | nx nifty jun                                  | nx nifty jun
CD-Certificate of Deposits                   | cd certificate of deposits                    | cd certificate of deposits
II NON-CONVERTIBLE DEBENTURES/BONDS           | in on convertible debentures bonds            | NONCONVERTIBLEDEBENTURESBONDS 


SyntaxError: invalid syntax (1784412931.py, line 3)

In [ ]:
ETF                                                                                       | etf
CMSI                                                                                    | cmsi
ReIT                                                                                   | reit
InvIT                                                                                 | invit
BUNLISTED                                                                         | unlisted
Securitized Debt Instruments                                   | Securitized Debt Instruments 
CNX NIFTY-JUN - - -                                                             | CNX NIFTY-JUN
CD-Certificate of Deposits                                        | CD-Certificate of Deposits
II NON-CONVERTIBLE DEBENTURES/BONDS                      | II NON-CONVERTIBLE DEBENTURES/BONDS


In [28]:
import wordninja
import re
from rapidfuzz import fuzz

SPECIAL_EXCLUSIONS = {"uncategorized", "unlisted"}
CUSTOM_WORDS = {"invit", "reit","securitised " , 'reits', "invits", "rtc", "cmsi", "etf"}

def is_abbreviation(word):
    return (
        word.isupper() or
        (len(word) <= 5 and not any(v in word.lower() for v in 'aeiou'))
    )

def clean_text(text):
    text = re.sub(r'[^a-zA-Z]', '', text)  # Remove non-alphabetic characters
    if is_abbreviation(text) or text.lower() in CUSTOM_WORDS:
        return text.lower()

    text = text.lower()
    tokens = [t for t in wordninja.split(text) if len(t) > 1]

    # Re-merge known custom compound words that might've been split
    merged_tokens = []
    i = 0
    while i < len(tokens):
        matched = False
        # Try to match 2 or more tokens together to form a known word
        for j in range(len(tokens), i, -1):
            candidate = ''.join(tokens[i:j])
            if candidate in CUSTOM_WORDS:
                merged_tokens.append(candidate)
                i = j
                matched = True
                break
        if not matched:
            merged_tokens.append(tokens[i])
            i += 1

    return " ".join(merged_tokens)


SPECIAL_EXCLUSIONS = {"uncategorized", "unlisted"}

def choose_final_label(cleaned_text):
    # Strip leading "b" if it's noise (like in BUNITS)
    tokens = cleaned_text.split()
    if tokens and tokens[0] == 'b':
        return " ".join(tokens[1:])
    return cleaned_text

def map_entries_to_labels(entries, threshold=95):
    cleaned_entries = {e: clean_text(e) for e in entries}
    categories = {}
    canonical_labels = {}

    for entry, cleaned in cleaned_entries.items():
        if cleaned in SPECIAL_EXCLUSIONS:
            categories[entry] = entry
            canonical_labels[entry] = entry
            continue

        matched = False
        for rep, rep_cleaned in canonical_labels.items():
            score = fuzz.ratio(cleaned, clean_text(rep_cleaned))
            if score >= threshold:
                categories[entry] = rep
                matched = True
                break

        if not matched:
            # New category
            label = choose_final_label(cleaned)
            categories[entry] = label
            canonical_labels[entry] = label

    return [(entry, cleaned_entries[entry], categories[entry]) for entry in entries]

entries = types.copy()
results = map_entries_to_labels(entries)

print(f"{'Original Entry':45} | {'Cleaned Entry':45} | Final Label")
print("-" * 120)
for orig, clean, label in results:
    print(f"{orig:45} | {clean:45} | {label}")

Original Entry                                | Cleaned Entry                                 | Final Label
------------------------------------------------------------------------------------------------------------------------
REIT/InvIT Instruments                        | reit invit instruments                        | reit invit instruments
Debt Instruments GOI                          | debt instruments goi                          | debt instruments goi
Corporate Debt Market Development Fund        | corporate debt market development fund        | corporate debt market development fund
Debt Instruments                              | debt instruments                              | debt instruments
Certificate of Deposit                        | certificate of deposit                        | certificate of deposit
Commercial Paper PUBA                         | commercial paper pub                          | commercial paper pub
Treasury Bill                                 | tre

In [9]:
import re
import wordninja
import nltk
from nltk.stem import WordNetLemmatizer
from rapidfuzz import fuzz

# Only needed once
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

SPECIAL_EXCLUSIONS = {"uncategorized", "unlisted"}

def is_abbreviation(word):
    return (
        word.isupper() or
        (len(word) <= 5 and not any(v in word.lower() for v in 'aeiou'))
    )
def clean_text(text):
    text = text.strip()

    # Abbreviation: skip all processing
    if is_abbreviation(text):
        return text

    # Step 1: Split compound tokens using wordninja
    split_words = wordninja.split(text.lower())

    # Step 2: Lemmatize non-abbreviations
    final_tokens = []
    for token in split_words:
        if is_abbreviation(token):
            final_tokens.append(token.upper())  # Preserve original form
        else:
            final_tokens.append(lemmatizer.lemmatize(token))

    return " ".join(final_tokens)

def choose_final_label(cleaned_text):
    tokens = cleaned_text.split()
    if tokens and tokens[0] == 'b':  # optional: drop noisy 'b' prefix
        return " ".join(tokens[1:])
    return cleaned_text

def map_entries(entries, threshold=95):
    cleaned_entries = {e: clean_text(e) for e in entries}
    categories = {}
    representatives = []

    for entry, cleaned in cleaned_entries.items():
        if cleaned in SPECIAL_EXCLUSIONS:
            categories[entry] = entry
            continue

        matched = False
        for rep in representatives:
            score = fuzz.ratio(cleaned, cleaned_entries[rep])
            if score >= threshold:
                categories[entry] = categories[rep]
                matched = True
                break

        if not matched:
            label = choose_final_label(cleaned)
            categories[entry] = label
            representatives.append(entry)

    return [(entry, cleaned_entries[entry], categories[entry]) for entry in entries]

entries = np.unique(types).copy()
results = map_entries(entries)

print(f"{'Original Entry':45} | {'Cleaned Entry':45} | Final Label")
print("-" * 120)
for orig, clean, label in results:
    print(f"{orig:45} | {clean:45} | {label}")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vaibh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vaibh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Original Entry                                | Cleaned Entry                                 | Final Label
------------------------------------------------------------------------------------------------------------------------
 Alternative Investment Fund Units            | alternative investment fund unit              | alternative investment fund unit
 Alternative Investment Funds                 | alternative investment fund                   | alternative investment fund
 Alternative Investment Funds AIF             | alternative investment fund a if              | alternative investment fund a if
 Amount Rs in Lakhs to NAV                    | amount RS in lakh to nav                      | amount RS in lakh to nav
 BOND & NCDs                                  | bond N CDS                                    | bond N CDS
 BOND & NCDs ICRA A                           | bond N CDS i CR a a                           | bond N CDS i CR a a
 BOND & NCDs Transport Infrastructure        

In [5]:
from rapidfuzz import fuzz
import re
import wordninja
from nltk.stem import PorterStemmer
import wordninja

entries = types

stemmer = PorterStemmer()

def is_abbreviation(word):
    return (
        word.isupper() or
        (len(word) <= 5 and not any(v in word.lower() for v in 'aeiou'))
    )

def clean_text(text):
    text = text.strip()

    if is_abbreviation(text):
        return text  # Preserve abbreviation

    # text = re.sub(r'[^a-zA-Z]', '', text).lower()

    # Step 1: Use wordninja to split compound words
    tokens = wordninja.split(text.lower())

    # Step 2: Apply stemming to individual tokens
    stemmed = [stemmer.stem(token) for token in tokens]

    return " ".join(stemmed)

def map_entries(entries, threshold=75):
    cleaned_map = {e: clean_text(e) for e in entries}
    categories = {}
    representatives = []

    for entry, cleaned in cleaned_map.items():
        matched = False
        for rep in representatives:
            score = fuzz.ratio(cleaned, cleaned_map[rep])
            if score >= threshold:
                categories[entry] = categories[rep]
                matched = True
                break
        if not matched:
            categories[entry] = cleaned  # First time seen → becomes its own label
            representatives.append(entry)

    return [(entry, cleaned_map[entry], categories[entry]) for entry in entries]

results = map_entries(entries)

print(f"{'Original Entry':40} | {'Cleaned Entry':40} | Final Label")
print("-" * 110)
for orig, clean, label in results:
    print(f"{orig:40} | {clean:40} | {label}")


Original Entry                           | Cleaned Entry                            | Final Label
--------------------------------------------------------------------------------------------------------------
REIT/InvIT Instruments                   | re it in vit instrument                  | re it in vit instrument
Debt Instruments GOI                     | debt instrument goi                      | debt instrument goi
Corporate Debt Market Development Fund   | corpor debt market develop fund          | corpor debt market develop fund
Debt Instruments                         | debt instrument                          | debt instrument goi
Certificate of Deposit                   | certif of deposit                        | certif of deposit
Commercial Paper PUBA                    | commerci paper pub a                     | commerci paper pub a
Treasury Bill                            | treasuri bill                            | treasuri bill
Equity & Equity related                 

In [8]:
import os 
import pandas as pd
import numpy as np
directory = "final_cleaned"
files = list(filter(lambda x : "~" not in x , os.listdir(directory)))

df = pd.DataFrame()
for file in files:
    df2 = pd.read_excel(os.path.join(directory, file))
    print(len(df2))
    df = pd.concat([df,df2])


296
5476
4592
666
1025
1986
1262
2676
2950
1445
2463
243
1635
8157
1701
886
615
5376
1992
1197
1502
4356
1295
5788
260
874
715
917
357
3153
312
1090
3153
229
1489
1781


In [9]:
len(df)

73910

In [10]:
from fuzzywuzzy import process
# Final dictionary with appropriate classification for ETFs as 'equity' and FOFs as 'others'

category_mapping = {
    # Equity & Equity Related
    'EQUITY & EQUITY RELATED': 'Equity & Equity Related',
    'Equity & Equity Related': 'Equity & Equity Related',
    'Equity & Equity related': 'Equity & Equity Related',
    'Equity & Equity Related Instruments': 'Equity & Equity Related',
    'Equity & Equity related Aerospace & Defense': 'Equity & Equity Related',
    'Equity & Equity related Automobiles': 'Equity & Equity Related',
    'Equity & Equity related Banks': 'Equity & Equity Related',
    'Equity & Equity related Biotechnology': 'Equity & Equity Related',
    'Equity & Equity related Capital Markets': 'Equity & Equity Related',
    'Equity & Equity related Construction': 'Equity & Equity Related',
    'Equity & Equity related Consumer Durables': 'Equity & Equity Related',
    'Equity & Equity related Electrical Equipment': 'Equity & Equity Related',
    'Equity & Equity related Exchange Traded Funds': 'Equity & Equity Related',
    'Equity & Equity related Finance': 'Equity & Equity Related',
    'Equity & Equity related IT - Software': 'Equity & Equity Related',
    'Equity & Equity related Industrial Products': 'Equity & Equity Related',
    'Equity & Equity related Pharmaceuticals & Biotechnology': 'Equity & Equity Related',
    'Equity & Equity related Realty': 'Equity & Equity Related',
    'Equity & Equity related Semiconductors': 'Equity & Equity Related',
    'Equity Shares': 'Equity & Equity Related',
    'Equity shares': 'Equity & Equity Related',
    'FOREIGNEQUITYSECURITIES': 'Equity & Equity Related',
    'Foreign Securities - Equity': 'Equity & Equity Related',
    'EQUITYEQUITYRELATED': 'Equity & Equity Related',

    # Exchange Traded Funds (ETFs)
    'Exchange Traded Fund': 'Exchange Traded Funds (ETFs)',
    'Exchange Traded Funds': 'Exchange Traded Funds (ETFs)',
    'EXCHANGE TRADED FUND UNITS': 'Exchange Traded Funds (ETFs)',
    'ETF': 'Exchange Traded Funds (ETFs)',
    'International Exchange Traded Funds': 'Exchange Traded Funds (ETFs)',
    'International Exchange Traded Funds Exchange Traded Funds': 'Exchange Traded Funds (ETFs)',
    'Foreign Securities and/or overseas ETF': 'Exchange Traded Funds (ETFs)',
    'Foreign Securities/Overseas ETFs': 'Exchange Traded Funds (ETFs)',
    'FOREIGNETF': 'Exchange Traded Funds (ETFs)',

    # Bonds & Debentures
    'BOND & NCDs': 'Bonds & Debentures',
    'Bonds': 'Bonds & Debentures',
    'Debentures and Bonds': 'Bonds & Debentures',
    'Zero Coupon Bond': 'Bonds & Debentures',
    'Zero Coupon Bonds': 'Bonds & Debentures',
    'Zero Coupon Bonds / Deep Discount Bonds': 'Bonds & Debentures',
    'Non Convertible Debentures': 'Bonds & Debentures',
    'Non Convertible Debentures / Bonds': 'Bonds & Debentures',
    'NON-CONVERTIBLE DEBENTURES/BONDS/ZCB': 'Bonds & Debentures',
    'II NON-CONVERTIBLE DEBENTURES/BONDS': 'Bonds & Debentures',
    'II NON-CONVERTIBLE DEBENTURES/BONDS/ZCB': 'Bonds & Debentures',
    'Convertible Debenture': 'Bonds & Debentures',
    'Compulsory Convertible Debenture': 'Bonds & Debentures',
    'COMPULSORILYCONVERTIBLEDEBENTURE': 'Bonds & Debentures',
    'Fixed rates bonds - Corporate': 'Bonds & Debentures',
    'Debt Instruments': 'Bonds & Debentures',
    'Debt Instruments SOVEREIGN': 'Bonds & Debentures',
    'DEBT INSTRUMENTS': 'Bonds & Debentures',
    'DEBTINSTRUMENTS': 'Bonds & Debentures',
    'Government Bonds': 'Bonds & Debentures',
    'Government Securities': 'Bonds & Debentures',
    'Government Securities / SDL': 'Bonds & Debentures',
    'Government Securities Central/State': 'Bonds & Debentures',
    'Government Securities Central/State Cash & Equivalent': 'Bonds & Debentures',
    'State Government Securities': 'Bonds & Debentures',
    'Govt Security': 'Bonds & Debentures',
    'Govt Securities / SDL': 'Bonds & Debentures',
    'Central Government Securities': 'Bonds & Debentures',
    'GOVERNMENT SECURITIES': 'Bonds & Debentures',
    'GOVERNMENTSECURITIES': 'Bonds & Debentures',
    'GOVERNMENTSECURITIESCENTRALSTATE': 'Bonds & Debentures',
    'i Government Securities': 'Bonds & Debentures',
    'ii State Government Securities': 'Bonds & Debentures',

    # Money Market Instruments
    'Certificate of Deposit': 'Money Market Instruments',
    'Certificate of Deposits': 'Money Market Instruments',
    'Certificate of Deposit IND A': 'Money Market Instruments',
    'Certificate of Deposits C': 'Money Market Instruments',
    'CERTIFICATEOFDEPOSIT': 'Money Market Instruments',
    'CERTIFICATEOFDEPOSITCD': 'Money Market Instruments',
    'CD-Certificate of Deposits': 'Money Market Instruments',
    'Commercial Paper': 'Money Market Instruments',
    'Commercial Papers': 'Money Market Instruments',
    'Commercial Papers C': 'Money Market Instruments',
    'COMMERCIALPAPER': 'Money Market Instruments',
    'COMMERCIALPAPERSCP': 'Money Market Instruments',
    'Treasury Bill': 'Money Market Instruments',
    'Treasury Bills': 'Money Market Instruments',
    'Treasury Bill/Cash Management Bill': 'Money Market Instruments',
    'Treasury Bill/Cash Management Bill CRISIL A': 'Money Market Instruments',
    'Treasury Bill/Cash Management Bill SOVEREIGN': 'Money Market Instruments',
    'TREASURYBILL': 'Money Market Instruments',
    'TREASURYBILLS': 'Money Market Instruments',
    'T-Bil': 'Money Market Instruments',
    'Tri Party Repo TREPs': 'Money Market Instruments',
    'TREPS / Reverse Repo': 'Money Market Instruments',
    'Reverse Repo / TREPS': 'Money Market Instruments',
    'Reverse Repo --': 'Money Market Instruments',

    # Alternative Investments
    'Alternative Investment Fund': 'Alternative Investments',
    'Alternative Investment Fund Units': 'Alternative Investments',
    'Alternative Investment Funds': 'Alternative Investments',
    'Alternative Investment Funds AIF': 'Alternative Investments',
    'Units of an Alternative Investment Fund AIF': 'Alternative Investments',
    'Investment in AIF': 'Alternative Investments',
    'AIF CAT': 'Alternative Investments',
    'ALTERNATIVEINVESTMENTFUND': 'Alternative Investments',
    'ALTERNATIVEINVESTMENTFUNDUNITS': 'Alternative Investments',
    'CDMDFAIF': 'Alternative Investments',
    'Infrastructure Investment Trusts': 'Alternative Investments',
    'Units of Infrastructure Investment Trusts InvITs': 'Alternative Investments',
    'InvIT': 'Alternative Investments',
    'Invits': 'Alternative Investments',
    'Units issued by REITs & InvITs': 'Alternative Investments',
    'REIT': 'Alternative Investments',
    'Reits': 'Alternative Investments',
    'ReIT': 'Alternative Investments',
    'Units of Real Estate Investment Trust REITs': 'Alternative Investments',
    'Real Estate Investment Trusts': 'Alternative Investments',
    'Real Estate Investment Trust': 'Alternative Investments',
    'UNITS OF INVIT': 'Alternative Investments',
    'UNITS OF REIT': 'Alternative Investments',
    'UNITSISSUEDBYINVIT': 'Alternative Investments',
    'UNITSISSUEDBYREIT': 'Alternative Investments',
    'BUNITSOFREALESTATEINVESTMENTTRUSTSREITS': 'Alternative Investments',

    # Other/Uncategorized
    'Others': 'Other/Uncategorized',
    'OTHERS': 'Other/Uncategorized',
    'UNLISTED': 'Other/Uncategorized',
    'Unlisted': 'Other/Uncategorized',
    'Privately Placed / Unlisted': 'Other/Uncategorized',
    'Privately Placed/Unlisted': 'Other/Uncategorized',
    'Privately placed / Unlisted': 'Other/Uncategorized',
    'Option wise per unit Net Asset Values are as follows': 'Other/Uncategorized',
    'ISINCODE securityname Yield to Call': 'Other/Uncategorized',
    'Arbitrage': 'Other/Uncategorized',
    'Numero Uno International Ltd Finance e-': 'Other/Uncategorized',
    'Margin Mutual Fund Units': 'Other/Uncategorized',
    'RISKLEVELBASEDONPORTFOLIOASONMAY': 'Other/Uncategorized',
    'RISKLEVELOFTIERBENCHMARKASONMAY': 'Other/Uncategorized',
    'CMSI': 'Other/Uncategorized',
    'CNX NIFTY-JUN - - -': 'Other/Uncategorized',
    'Amount Rs in Lakhs to NAV': 'Other/Uncategorized',
    'uncategorised': 'Other/Uncategorized'
}
from fuzzywuzzy import process
keys = list(category_mapping.keys())
def fuzzy_map_entries_to_category(entries):

    results = {}

    for entry in entries:
        match, score = process.extractOne(entry, keys)
        category = category_mapping.get(match, "others")
        results[entry] = category
    
    return results
mapped_entries = fuzzy_map_entries_to_category(np.unique(df["Type"]))

c:\Users\vaibh\miniconda3\envs\fixedDeposit\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [11]:
df["Type"] = df["Type"].map(mapped_entries)

In [12]:
df = df.drop("yield to call (ytc)", axis = 1)

In [13]:
new_cols = ["Name of Instrument",	"ISIN" , "Coupon" , "Industry",	"Quantity", "Market Value",	"% to Net Assets","Yield","Type","AMC","Scheme Name" , "Scheme ISIN" ]
df.columns = new_cols

In [14]:
df.to_csv("DIRECT_MAY_combined_sheet_reduced_types.csv",index = False)

In [15]:
temp = df[df["ISIN"] == "INE398R01022"]

In [16]:
for (idx , row) in temp.iterrows():
    print(row)

Name of Instrument          Syngene International Limited 
ISIN                                          INE398R01022
Coupon                                                 0.0
Industry                               Healthcare Services
Quantity                                          165000.0
Market Value                                       1066.81
% to Net Assets                                   0.058604
Yield                                                  0.0
Type                               Equity & Equity Related
AMC                   ADITYA BIRLA SUN LIFE ARBITRAGE FUND
Scheme Name              Aditya Birla Sun Life Mutual Fund
Scheme ISIN                                   INF209K01P80
Name: 2202, dtype: object
Name of Instrument                Syngene International Limited 
ISIN                                                INE398R01022
Coupon                                                       0.0
Industry                                     Healthcare Services
Quanti

In [20]:
df[df["Scheme Name"].apply(lambda x : "ppfas" in x.lower())]["Scheme ISIN"].unique()

array(['INF879O01019', 'INF879O01142', 'INF879O01092', 'INF879O01191',
       'INF879O01217', 'INF879O01233'], dtype=object)

In [67]:
df[df["ISIN"].apply( lambda x : "INE153T01027" in x )]

,Name of Instrument,ISIN,Coupon,Industry,Quantity,Market Value,% to Net Assets,Yield,Type,AMC,Scheme Name,Scheme ISIN
436,BLS International Services Limited BLSS01,INE153T01027,0.0,Leisure Services,1544.0,6.2771,0.020000,0.0,Equity & Equity Related,Axis Nifty 500 Index Fund,Axis Mutual Fund,INF846K019W9
1593,BLS International Services Limited,INE153T01027,0.0,Leisure Services,108629.0,441.6300,1.730000,0.0,Equity & Equity Related,DSP Nifty Smallcap250 Quality 50 Index Fund,DSP Mutual Fund,INF740KA1TX9
166,BLS International Services Ltd.,INE153T01027,0.0,Leisure Services,20219.0,82.2000,0.350000,0.0,Equity & Equity Related,edelweiss nifty500 multicap momentum quality 5...,Edelweiss Mutual Fund,INF754K01TG9
988,BLS International Services Ltd.,INE153T01027,0.0,Leisure Services,2319.0,9.4300,0.350000,0.0,Equity & Equity Related,edelweiss nifty500 multicap momentum quality 5...,Edelweiss Mutual Fund,NaN
2061,BLS International Services Ltd.,INE153T01027,0.0,Leisure Services,8638.0,35.1200,0.260000,0.0,Equity & Equity Related,edelweiss nifty smallcap 250 index fund as on ...,Edelweiss Mutual Fund,INF754K01QU6
2656,BLS International Services Ltd.,INE153T01027,0.0,Leisure Services,7044.0,28.6400,0.960000,0.0,Equity & Equity Related,edelweiss bse internet economy index fund as o...,Edelweiss Mutual Fund,INF754K01UY0
45,BLS International Services Ltd.,INE153T01027,0.0,Leisure Services,40399.0,164.2400,0.540000,0.0,Equity & Equity Related,Helios Balanced Advantage Fund (An open-ended ...,Helios Mutual Fund,INF0R8701152
133,BLS International Services Ltd. 100565,INE153T01027,0.0,Leisure Services,649861.0,2642.0100,0.820000,0.0,Equity & Equity Related,Helios Flexi Cap Fund (An open-ended dynamic e...,Helios Mutual Fund,INF0R8701053
231,BLS International Services Ltd.,INE153T01027,0.0,Leisure Services,27688.0,112.5700,1.150000,0.0,Equity & Equity Related,Helios Mid Cap Fund (Mid Cap Fund - An open-en...,Helios Mutual Fund,INF0R8701335
1047,BLS International Services Ltd.,INE153T01027,0.0,Leisure Services,2209.0,8.9600,0.026088,0.0,Equity & Equity Related,ICICI Prudential BSE 500 ETF,ICICI Prudential Mutual Fund,INF109KC1V91


In [7]:
import os
import pandas as pd
directory = "final_cleaned"

files = list(filter(lambda x : "~" not in x , os.listdir(directory)))
for file in files:
    df = pd.read_excel(os.path.join(directory, file))
    print( f"amc:{file.split(".")[0]} , \t {df[df.iloc[:,7]!=0].iloc[:20,7].values.tolist()}")
       

amc:360 One Asset Management , 	 []
amc:Aditya Birla Sun Life Mutual Fund , 	 [6.7507, 6.757199999999999, 6.84, 6.775, 6.800000000000001, 6.694999999999999, 6.940612, 6.930698, 6.988301, 6.988301, 6.966998, 6.791493, 6.980217, 6.993487, 6.360422000000001, 6.373065, 6.518301999999999, 6.871204000000001, 6.754585000000001, 6.783846]
amc:Axis Mutual Fund , 	 []
amc:Bandhan Mutual Fund , 	 []
amc:Bank of India Mutual Fund , 	 []
amc:Baroda BNP Paribas Mutual Fund , 	 []
amc:Canara Robeco Mutual Fund , 	 [7.15, 7.07, 7.03, 7.139999999999999, 6.600000000000001, 5.61, 5.61, 5.6, 6.39, 6.93, 6.370000000000001, 6.09, 6.529999999999999, 6.54, 6.69, 6.84, 7.139999999999999, 6.709999999999999, 6.480000000000001, 6.600000000000001]
amc:DSP Mutual Fund , 	 []
amc:Edelweiss Mutual Fund , 	 [5.8113, 5.8378, 6.157, 6.1106, 6.157, 6.67, 6.6411, 6.6425, 6.475000000000001, 6.6425, 6.529999999999999, 6.5887, 6.45, 6.67, 6.5067, 6.5175, 6.486699999999999, 6.905200000000001, 6.5887, 6.460000000000001]
amc:Fr